# pyaegean — one line of Homer, all the way down

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/getting-started.ipynb)

**pyaegean** is a specialist toolkit for Ancient Greek and the Aegean syllabic scripts (Linear A, Linear B, Cypriot, and Cypro-Minoan). Instead of touring its functions in a vacuum, this notebook does what a scholar actually does: sit down with real text and read it.

We take **one** famous line — the opening of Homer's *Odyssey* — and walk it down the whole linguistic ladder, one small composable function at a time:

> text → syllables → accent → **metre** → part-of-speech → **morphology** → dictionary meaning → **dependency tree**

Then we turn the same toolkit on an *undeciphered* script, Linear A, where it gives **leads for a human**, never answers.

**How this notebook is staged**
- **Act 1 & 3 (the core) run 100% offline** — no downloads, no API key. They complete in seconds and are the spine of the tutorial.
- **Act 2 (opt-in)** switches on four heavier backends that *download data or train a model*. Every one is marked ⬇️ **HEAVY (OPTIONAL)** with its cost and gated behind a single `RUN_HEAVY` switch — the notebook runs top-to-bottom and is complete **without** them.

> **All Linear A interpretation is EXPLORATORY.** The script is undeciphered; these tools surface *leads for a human expert*, never ground truth. The library labels this everywhere, and so do we.

New to Python? You don't need to install anything — press **Open in Colab** above and run each cell top to bottom with Shift+Enter.

In [ ]:
# pyaegean works fully offline once installed. This installs it the first time
# you open the notebook in Colab; locally it's a no-op if it's already present.
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)
print("scripts :", aegean.registered_scripts())
# ['cypriot', 'cyprominoan', 'greek', 'lineara', 'linearb'] — all four Aegean scripts + Greek

# Tip: pyaegean's objects (Corpus, Document, analysis results) render as rich
# tables/cards in a notebook. Outside Jupyter they're ordinary Python values.

In [ ]:
# ── The one switch for the optional, downloading backends ────────────────
# Leave this False to keep the whole notebook offline. Flip it to True (on decent
# wifi) to run the heavy cells in Acts 2 and 4:
#   * use_treebank()           ~15 MB prebuilt bundle -- attested lemmas + gold POS/morphology
#   * use_neural_lemmatizer()  ~232 MB download -- seq2seq lemmas for UNSEEN forms (76.3%)
#   * use_lsj()                ~15 MB prebuilt index  -- full Liddell-Scott-Jones glossing
#   * use_parser()             prebuilt model (in the same ~15 MB bundle as the treebank)
#   * aegean.load('damos')     ~2 MB -- the full Linear B corpus (Act 4)
# The treebank/LSJ/parser artifacts are small project-hosted prebuilt assets; they are
# built locally from the 75/270 MB upstream sources only if an asset is unreachable.
# Each fetches once, then caches. Every other cell runs offline with no downloads.
RUN_HEAVY = False

print("Heavy/optional cells are",
      "ON — first run downloads." if RUN_HEAVY
      else "OFF — the offline core still runs in full.")

## Act 1 · Reading one line of Homer (offline)

The Greek pipeline is a set of **small, independent functions** you compose yourself — and the whole of this act is **offline**, no API key. We'll take Homer's opening line and read it the way a philologist does.

> **No Greek keyboard?** Type **Beta Code** (the TLG/Perseus ASCII convention) and convert. Round-trips are NFC-safe, so you can enter any line below from a normal keyboard.

In [ ]:
from aegean import greek

# Beta Code in, Unicode out (and back) — verified round-trips:
print(greek.betacode_to_unicode("a)/ndra"))   # ἄνδρα
print(greek.unicode_to_betacode("λόγος"))     # lo/gos   (context-sensitive final ς)

LINE = "ἄνδρα μοι ἔννεπε, Μοῦσα, πολύτροπον, ὃς μάλα πολλὰ"   # Odyssey 1.1
print(greek.tokenize_words(LINE))
# ['ἄνδρα', 'μοι', 'ἔννεπε', 'Μοῦσα', 'πολύτροπον', 'ὃς', 'μάλα', 'πολλὰ']

### Rungs 1–2 · Syllables and accent

Syllabification respects diphthongs, *muta cum liquida* clusters, and doubled consonants. Accent analysis classifies the word in the traditional scheme (oxytone / paroxytone / properispomenon …) and gives you structured fields, not just a string.

In [ ]:
print(greek.syllabify("ἄνδρα"))         # ['ἄν', 'δρα']
print(greek.syllabify("πολύτροπον"))    # ['πο', 'λύ', 'τρο', 'πον']

info = greek.accentuation("Μοῦσα")
print(info.accent_type, info.position_from_end, info.classification)
# circumflex 2 properispomenon  — a circumflex on a long penult

### Rung 3 · Scan the metre

The *Odyssey* is dactylic hexameter. The scanner resolves each syllable's quantity **in context** — across word boundaries, applying *correptio*, treating muta-cum-liquida as the genuine ambiguity it is — then reports the feet and the **caesura** (the line's main pause). Notation you'll know from any commentary: **—** heavy, **⏑** light, **×** *anceps* (the "either" final syllable).

In [ ]:
sc = greek.scan_hexameter(LINE)
print(sc.pattern)                  # —⏑⏑|—⏑⏑|—⏑⏑|—⏑⏑|—⏑⏑|—×   (five dactyls)
print([f.name for f in sc.feet])   # ['dactyl','dactyl','dactyl','dactyl','dactyl','final']
print("caesura:", sc.caesura, "before syllable", repr(sc.syllables[sc.caesura_index]))
# caesura: trochaic before syllable 'πο'

### An honest scanner

When a line only scans if two written vowels merge into one syllable (*synizesis* — as in *Iliad* 1.1, where `Πηληϊάδεω` must contract), the scanner **declines rather than forcing a fit**: it raises `ScansionError`. A tool that tells you when it *can't* is one you can trust when it does. To see where a line is genuinely ambiguous *before* a metre is imposed, ask `syllable_options`.

In [ ]:
try:
    greek.scan_hexameter("μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος")   # Iliad 1.1
except greek.ScansionError as e:
    print("declined:", str(e)[:62], "…")
# declined: line does not scan as dactylic hexameter (17 syllables): …

# Inspect a syllable's *possible* quantities (πα before muta-cum-liquida: either):
print(greek.syllable_options("πατρός"))
# [('πα', ['heavy', 'light']), ('τρός', ['light'])]

### Optional aside · How did it *sound*?

`to_ipa` gives a reconstructed transcription for two periods — `"attic"` (Classical, default) and `"koine"` (Hellenistic). **Reconstructed and approximate**: several values (vowel quality, the date of iotacism) are scholarly judgement calls.

In [ ]:
print(greek.to_ipa("θεός"))           # tʰeos   (Attic: aspirated θ, voiceless)
print(greek.to_ipa("θεός", "koine"))  # θeos    (Koine: θ has fricativised)
print(greek.to_ipa("καί", "koine"))   # ke      (Koine iotacism: αι → /e/)

### Rung 4 · Parts of speech and morphology — meeting the baseline honestly

The **offline baseline** is rule-based and deliberately honest. Closed-class words (article, prepositions, conjunctions, particles, pronouns, the copula) are tagged reliably from a lexicon; open-class words fall back to a light suffix heuristic, and `analyze` returns *every* reading an ending licenses — Greek inflection is richly ambiguous, and that ambiguity is the linguistic reality.

Watch it tag the verb `ἔννεπε` and the adverb `μάλα` as **NOUN**, and reconstruct the *wrong, unaccented* lemma for `ἄνδρα` with `lemma_certain=False`. That's the baseline being honest, not broken — and the cliffhanger Act 2 will pay off.

In [ ]:
# Baseline POS on our line — honest about its limits:
for w, p in greek.pos_tags(LINE):
    print(f"  {w:11} {p}")
# ἔννεπε / μάλα fall back to NOUN; the closed-class ὃς is correctly PRON.

# A REGULAR form analyses cleanly — all the readings the -ον ending licenses:
print("\nλόγον readings:")
for a in greek.analyze("λόγον"):
    print("  ", a)
# λόγος [NOUN acc sg masc] / [acc sg fem] / [nom sg neut] / [acc sg neut] / [voc sg neut]

# ἄνδρα is the acc. sg. of the irregular 3rd-declension ἀνήρ — beyond the rules.
# The lemma comes back UNACCENTED and flagged uncertain: the engine's "I guessed":
print("\nἄνδρα on the baseline:")
for a in greek.analyze("ἄνδρα"):
    print("  ", a, "| certain:", a.lemma_certain)
# ανδρα [NOUN ...] | certain: False   ← unaccented + wrong; ἄνδρα is acc sg of ἀνήρ
# The lesson: read `lemma_certain`. The fix for attested forms is one switch away →

## Act 2 · The payoff: opt-in backends that turn guesses into scholarship

Everything above ran **offline in seconds**. pyaegean offers **five opt-in** backends built from gold scholarly data: three resolve *attested* forms (treebank, the dictionary registry, parser), one — the neural lemmatizer — *generates* lemmas for **unseen** forms, and one — the **neural pipeline** — does the whole job (tagging, morphology, parsing, lemmas) with one jointly-trained model, at state-of-the-art accuracy on the standard Ancient Greek benchmarks. Each is one function call, then cached, but each downloads data or trains a model on first use, so they're isolated here and gated behind `RUN_HEAVY`.

| Backend | Call | Gives you | First-run cost |
|---|---|---|---|
| **Treebank** (AGDT v2.1, CC BY-SA) | `greek.use_treebank()` | attested accented lemmas + gold POS/morphology | ~75 MB download |
| **Neural lemmatizer** (GreTa seq2seq, `[neural]`) | `greek.use_neural_lemmatizer()` | generated lemmas for *unseen* forms (76.3%) | ~232 MB download |
| **Dictionary registry** (LSJ, Middle Liddell, Cunliffe, Abbott-Smith) | `greek.use_lsj()` / `greek.use_lexicon(id)` | glosses from any dictionary, plus Logeion deep-links | ~2–15 MB per dictionary |
| **Dependency parser** (arc-eager + perceptron, trained on AGDT) | `greek.use_parser()` | dependency trees with Prague labels | ~2-3 min train, then ~4 MB |
| **Neural pipeline** (joint model, `[neural]`) | `greek.use_neural_pipeline()` | POS + morphology + **UD trees** + lemmas in one pass — the package's most accurate option (97.0 UPOS / 94.3 lemma / 90.2 UAS, UD Perseus test) | ~518 MB download |

Each is fetched to your cache, **never bundled**, and has a matching `disable_*()`. The cells below are guarded by `RUN_HEAVY`: if it's `False` they print what they *would* do and skip the download, and the rest of the notebook still works.

> ⬇️ **HEAVY (OPTIONAL) — Treebank backend.** First run fetches the **~15 MB prebuilt AGDT bundle** (built locally from the ~75 MB treebank only if the asset is unreachable); afterwards it's instant. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the prebuilt treebank lexicon (~15 MB).")
else:
    greek.use_treebank()        # one-time ~15 MB prebuilt fetch, then cached

    # The same line now tags the open-class words correctly:
    print("POS  :", greek.pos_tags(LINE)[:5])
    # [('ἄνδρα','NOUN'), ('μοι','PRON'), ('ἔννεπε','VERB'), (',','PUNCT'), ('Μοῦσα','NOUN')]

    # The ἄνδρα cliffhanger, resolved — attested, correctly accented:
    print("ἄνδρα →", greek.lemmatize("ἄνδρα"))     # ἀνήρ
    for w in ("ἔφη", "γυναικός", "πόλεως"):
        print(f"  {w:9} → {greek.lemmatize(w)}")    # φημί · γυνή · πόλις

    # Put a NUMBER on the lift, on a hand-authored independent gold set:
    from aegean.greek import benchmark
    benchmark.compare_modes()   # toggles the backend internally, leaving it disabled
    # baseline : lemma  28% (5/18)   · pos  50% (10/20)
    # treebank : lemma 100% (18/18)  · pos 100% (20/20)

    greek.use_treebank()        # re-enable it for the cells below (instant once cached)

### The unseen-form lever · the neural lemmatizer

The treebank aces *attested* forms — but on a genuinely **unseen** word it falls back to the rule baseline (which, as we saw with `ἄνδρα`, often returns the wrong, unaccented lemma). pyaegean's `[neural]` backend closes that gap: a **GreTa** (Ancient-Greek T5) seq2seq, exported to int8 ONNX and run **torch-free** (numpy + onnxruntime), that *generates* the lemma. It reaches **76.3% on unseen forms** — the hardest case, where recovering a lemma means an internal stem/accent change, not just a suffix swap. It ships as a **hybrid**: the bundled gold lookup answers seen forms instantly, the seq2seq handles the rest, and `greek.lemmatize` cascades treebank → neural → edit-tree → seed.

> ⬇️ **HEAVY (OPTIONAL) — neural lemmatizer.** First run downloads the **~232 MB** ONNX model (the `[neural]` extra) and caches it; afterwards it's instant. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to download the neural lemmatizer (~232 MB, the [neural] extra).")
else:
    %pip install -q "pyaegean[neural]"   # onnxruntime + tokenizers (no torch)
    greek.use_neural_lemmatizer()        # one-time ~232 MB model download, then cached

    # Unseen, inflected forms the seed table can't reach — the seq2seq *generates* the lemma:
    for w in ("ἐποίησεν", "ἔγραψεν", "ἀπέθανεν"):
        print(f"  {w:11} → {greek.lemmatize(w)}")
    # e.g. ἐποίησεν → ποιέω   (generated, not a table lookup)

    greek.disable_neural_lemmatizer()    # back to the offline cascade

### What does *λόγος* actually mean?

> ⬇️ **HEAVY (OPTIONAL) — dictionaries.** First run fetches a small prebuilt index per dictionary (the LSJ index is ~15 MB; Middle Liddell / Cunliffe / Abbott-Smith are 0.1–2.3 MB). Runs only if `RUN_HEAVY = True`.

`use_lsj()` is one backend in a **lexicon registry**. `greek.use_lexicon(id)` activates any of **Middle Liddell** (concise classical), **Cunliffe** (Homeric), **Abbott-Smith** (New Testament), or **LSJ**, and `greek.gloss(word, dictionary=id)` glosses from the one you pick. Glossing **composes** with the lemmatizer: hand it an inflected form and it tries the form, then lemmatizes (using the treebank above if you switched it on) and retries — so `ἀνδρός` resolves to *ἀνήρ* automatically. For dictionaries pyaegean does not host (Autenrieth, Slater, …), `greek.lexicon_link(word)` builds a Logeion deep-link, and that one runs **offline**.

In [ ]:
# lexicon_link is OFFLINE — a Logeion deep-link to a word's dictionary entry (lemmatized first):
print("Logeion:", greek.lexicon_link("μῆνις"))   # a (percent-encoded) link to logeion.uchicago.edu/μῆνις

# The rest fetches small dictionary indexes; RUN_HEAVY gates it (flip the switch at the top).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the dictionary indexes (0.1–15 MB each).")
else:
    greek.use_treebank()                  # so inflected forms (ἀνδρός → ἀνήρ) resolve for glossing
    greek.use_lsj()                       # one-time ~15 MB prebuilt fetch, then cached
    greek.use_lexicon("middle-liddell")   # the concise Intermediate Lexicon
    greek.use_lexicon("cunliffe")         # Cunliffe, A Lexicon of the Homeric Dialect
    print("dictionaries:", [i.id for i in greek.lexica()])

    for w in ("ἀνδρός", "βάλλω"):
        print(f"{w:7} → {greek.gloss(w)[:60]}")   # LSJ by default; ἀνδρός is lemmatised to ἀνήρ

    # The same word from different dictionaries — choose one with dictionary=:
    print("\nμῆνις, three ways (each glosses it as 'wrath' in its own register):")
    for d in ("lsj", "middle-liddell", "cunliffe"):
        print(f"  {d:14} {greek.gloss('μῆνις', dictionary=d)[:55]}")

    entry = greek.lookup("λόγος")          # the full structured LSJ entry
    print("\nλόγος —", len(entry.senses), "senses; the first:", entry.senses[0].text[:42])
    entry   # renders as a tidy definition card in Jupyter

### Three more levers · inflection, register, rarity

The same gold data drives three more opt-in tools. **Inflection synthesis** runs lemmatization *backwards*: `greek.inflect` maps a lemma plus a feature spec to the attested form(s) in the AGDT, and `greek.paradigm` lists the whole paradigm. **`greek.usage`** reads a word's dialect and register markers off its LSJ entry. And **`greek.terminology_rarity`** scores how unusual each word is *relative to a reference corpus* (here the Greek NT), a translation-difficulty signal that surfaces rare or technical vocabulary.

> ⬇️ **HEAVY (OPTIONAL).** Reuses the treebank, LSJ, and NT backends. Runs only if `RUN_HEAVY = True`. `usage` and `rarity` are **EXPLORATORY** aids (heuristic markers, corpus-relative scores), not measured truth.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True for inflection synthesis, usage tags, and rarity.")
else:
    greek.use_treebank()        # the inverse index + rarity's lemmatizer both lean on it

    # 1. Inflection synthesis — the lemmatizer run BACKWARDS (attested AGDT forms):
    greek.use_inflector()                                   # one-time inverse-index build, then cached
    print("λόγος, gen sg →", greek.inflect("λόγος", case="gen", number="sg"))   # ('λόγου',)
    print("paradigm cells:", len(greek.paradigm("λόγος")), "— e.g. acc sg",
          greek.inflect("λόγος", case="acc", number="sg"))                       # 14 — e.g. ('λόγον',)

    # 2. Dialect / register, read off the LSJ entry (heuristic, EXPLORATORY):
    greek.use_lsj()
    u = greek.usage("ἄναξ")
    print("ἄναξ →", "dialects:", u.dialects or "—", "| registers:", u.registers)
    # ἄναξ → dialects: — | registers: ('tragic', 'comic', 'poetic')  — a poetic word for 'lord'

    # 3. Terminology rarity, RELATIVE to the Greek NT — the rare word stands out:
    nt = greek.load_nt()
    r = greek.terminology_rarity("τὸν ἄρτον ἡμῶν τὸν ἐπιούσιον δὸς ἡμῖν", nt)   # Matt 6:11
    print("rarity:", round(r.overall, 3), "| hardest:",
          [(w.word, w.label) for w in r.hardest(2)])
    # rarity: 0.334 | hardest: [('ἐπιούσιον', 'absent'), ('ἄρτον', 'common')]
    # ἐπιούσιον — the famous near-hapax 'daily' — surfaces as the hardest word to render.

### Rung 5 (the last) · Seeing the syntax

> ⏱️ **HEAVY (OPTIONAL) — Dependency parser.** First run fetches the **prebuilt model** from the shared AGDT bundle (trains ~2-3 min locally, pure Python, only if the asset is unreachable). Runs only if `RUN_HEAVY = True`.

This is an **honest baseline**, not a research-grade parser. Ancient Greek is richly *non-projective* (only ~31% of AGDT sentences are projective) and arc-eager builds only projective trees. It gives clean trees for main-clause syntax, with the gold **AGDT/Prague** labels (SBJ, OBJ, PRED, ATR, ADV, Aux…). We'll parse the opening of John (a clean main clause), and `greek.evaluate_parser()` reports the real accuracy so you're never misled.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the prebuilt parser model.")
else:
    greek.use_treebank()        # optional — better POS/lemmas feed the parser
    greek.use_parser()          # prebuilt model on first use (local train is the fallback)

    tree = greek.parse("ἐν ἀρχῇ ἦν ὁ λόγος")   # John 1:1a
    print(tree)
    # 1 ἐν     ADP  AuxP ->3(ἦν)
    # 2 ἀρχῇ   NOUN ADV  ->1(ἐν)
    # 3 ἦν     VERB PRED ->0(ROOT)
    # 4 ὁ      DET  ATR  ->5(λόγος)
    # 5 λόγος  NOUN SBJ  ->3(ἦν)
    print("root:", tree.root().form,
          "| dependents of ἦν:", [t.form for t in tree.children(3)])  # ἦν | ['ἐν', 'λόγος']

    # The honest scorecard on held-out AGDT:
    print(greek.evaluate_parser())
    # ~0.51 UAS / 0.42 LAS across all text (~0.67 / 0.57 on the projective subset)

**Where we are.** From a bare line of epic Greek we read it, scanned the metre, met an honest baseline — and then, with three one-line opt-ins, got attested lemmas, dictionary senses, and a syntax tree. Each backend has a matching `greek.disable_treebank()` / `disable_lsj()` / `disable_parser()` to drop back to the pure-offline core. Network is needed only on the first call of each.

## Act 3 · The other side: Linear A, where the tools give *leads* (offline, EXPLORATORY)

Same researcher, opposite problem. **Linear A is undeciphered** — we can transliterate the signs but we don't know the language. So pyaegean's Aegean analysis never claims answers: every method here is labelled **EXPLORATORY** and surfaces patterns worth a human's attention.

We'll do one concrete thing — check whether a 3,500-year-old account *adds up* — then let the corpus suggest its own structure. The bundled corpus is the Linear A Workbench dataset (transliterated, with numerals and find-spot metadata) and loads **offline**.

In [ ]:
corpus = aegean.load("lineara")        # 1,721 inscriptions, bundled, offline
print(len(corpus), "inscriptions")

doc = corpus.get("HT13")               # a well-known account from Haghia Triada
print("words   :", [t.text for t in doc.words])
print("numerals:", [t.text for t in doc.numerals])
print("site    :", doc.meta.site, "| period:", doc.meta.period)   # Haghia Triada | LMIB
doc   # in Jupyter the Document renders as a glyph/transcription card

### Does the arithmetic close?

Many Linear A tablets are accounts: line items followed by a total word, **KU-RO**. `balance_check` sums the items a total governs and compares them to the stated total.

> **EXPLORATORY.** Section boundaries are heuristic and Linear A metrology is genuinely contested. This *finds* lines worth a human's attention — it is not a verdict.

In [ ]:
from aegean.analysis import balance_check

for chk in balance_check(doc):
    print(chk)
# BalanceCheck(stated_total=130.5, computed_sum=131.0, item_count=6,
#              difference=0.5, balances=False, marker='KU-RO', total_line_index=7)
#
# The items sum to 131.0 but the scribe wrote 130.5 — a ½-unit gap. Ancient error?
# misread sign? an artefact of where *we* drew the section? A lead for a human.

### How does the total-word behave across the corpus?

Two composable tools. The **sign-pattern** language searches by shape (`*` = *exactly one sign*), so `KU-*-RO` matches a three-sign word — and `KU-RO` itself does **not** match (no middle sign). The **query engine** then combines surface conditions with `and`/`or`/`not`. (Unlike the interpretive tools, these predicates are deterministic filters, not exploratory.)

In [ ]:
from aegean.analysis import word_matches_sign_pattern, FilterRow, run_query

# Words shaped KU-?-RO (one sign between):
print([(w, c) for w, c in corpus.word_frequencies()
       if word_matches_sign_pattern(w, "KU-*-RO")])
# [('KU-MA-RO', 1)]

# Haghia Triada tablets (id contains 'HT') that contain the exact word KU-RO:
res = run_query(corpus, [
    FilterRow("id-contains", "HT"),
    FilterRow("ins-contains-word", "KU-RO", connector="and"),
], output="inscriptions")
print(len(res.inscriptions), "tablets:", [d.id for d in res.inscriptions][:8])
# 32 tablets: ['HT9a','HT9b','HT11a','HT11b','HT13','HT25b','HT27a','HT39']

### A minimal pair? KU-RO vs KI-RO

The second-most-common accounting word is **KI-RO** (often read as a *deficit/owed* counterpart to the KU-RO total). They differ in one vowel — pyaegean can make that intuition quantitative with a **weighted phonetic edit distance** (0 = identical → 1) and a per-phoneme **alignment**.

> **EXPLORATORY.** The phonetic values come from the conventional sign→sound transliteration of an *undeciphered* script. A small distance is a *lead*, never a sound law — which is exactly why the scheme is configurable.

In [ ]:
from aegean.scripts.lineara.phonetic import word_to_phonetic
from aegean.analysis import phonetic_distance, align_phonetic

a, b = word_to_phonetic("KU-RO"), word_to_phonetic("KI-RO")
print(a, b)                                  # kuro kiro
print("distance:", round(phonetic_distance(a, b), 3))   # 0.075  (one cheap vowel swap)

for cell in align_phonetic(a, b):
    print(f"  {cell.a or '—'}  {cell.b or '—'}  {cell.op}")
#   k  k  match
#   u  i  sub-vowel    ← the single, low-cost difference
#   r  r  match
#   o  o  match

### Letting the corpus suggest morphology

With no known grammar, we can still ask a purely distributional question: which words share a stem and differ by a **productive ending** (a final sign-string that recurs across many distinct words)? `find_morphological_clusters` answers it. The top cluster is the famous libation-formula family **JA-SA-SA-RA-ME**.

> **EXPLORATORY.** A "suffix" here is a recurring final sign-string, **not** a confirmed morpheme. These are candidate paradigms for a morphologist to weigh.

In [ ]:
from aegean.analysis import find_morphological_clusters

clusters = find_morphological_clusters(corpus.word_frequencies())
print(len(clusters), "candidate clusters")     # 81

fam = clusters[0]                               # the JA-SA family
print("stem:", fam.stem, "| total occurrences:", fam.total_count)
for m in fam.members:
    print(f"   {m.word:<16} x{m.count:<3} +{m.suffix or '(stem)'}")
# stem: JA-SA | total occurrences: 16
#    JA-SA-SA-RA-ME   x7   +SA-RA-ME
#    JA-SA            x4   +(stem)   ... etc.

### Do KU-RO and KI-RO travel together?

If KI-RO is the deficit counterpart to the KU-RO total, they should **co-occur** on the same tablets more than chance. pyaegean ships the corpus-linguistics association tests — log-likelihood ratio (Dunning G²) and Fisher's exact — over a 2×2 contingency table.

> **EXPLORATORY.** A strong association is a distributional fact worth explaining — it does not by itself confirm the semantic reading.

In [ ]:
from aegean.analysis import log_likelihood_ratio_2x2, fishers_exact

N = len(corpus)
presence = [{t.text for t in d.words} for d in corpus]
count = lambda w: sum(w in s for s in presence)

ca, cb = count("KU-RO"), count("KI-RO")
joint = sum(("KU-RO" in s and "KI-RO" in s) for s in presence)
print(f"docs: KU-RO={ca}  KI-RO={cb}  both={joint}  (of {N})")   # 34  12  5  (of 1721)
print("Dunning G² :", round(log_likelihood_ratio_2x2(joint, ca, cb, N), 2))  # 23.94
print("Fisher  p  :", f"{fishers_exact(joint, ca, cb, N):.2e}")               # 1.60e-06
# A strong, very-unlikely-by-chance association — a real lead for the deficit reading.

## Act 4 · Scale: the full Linear B corpus, one call away

The acts above worked on one line and on the bundled corpora. The same toolkit loads the **complete Mycenaean corpus** — DAMOS, the Database of Mycenaean at Oslo (~5,900 tablets, CC BY-NC-SA 4.0, fetched ~2 MB to your cache) — and the corpus statistics answer real questions about it. The classic: *what vocabulary makes Pylos different from the rest of the Mycenaean world?*

> ⬇️ **HEAVY (OPTIONAL)** — fetches ~2 MB on first run. Runs only if `RUN_HEAVY = True`.

In [ ]:
# RUN_HEAVY gates this cell (False by default — flip the one switch at the top to run it).
if not RUN_HEAVY:
    print("[skipped] set RUN_HEAVY = True to fetch the DAMOS corpus (~2 MB).")
else:
    from aegean.analysis import keyness

    damos = aegean.load("damos")                  # ~5,900 tablets, then cached
    pylos = damos.filter(site="Pylos")
    rest = [d for d in damos.documents if d.meta.site != "Pylos"]

    for row in keyness(pylos, rest)[:5]:          # G² + log-ratio, strongest first
        print(f"{row.item:10}  G²={row.log_likelihood:6.1f}  "
              f"log-ratio={row.log_ratio:+.1f}  ({row.target_count}× at Pylos)")
    # pe-mo ('seed'), o-na-to ('lease plot'), ko-to-na ('land plot') … — the Pylos
    # land-tenure series surfaces immediately: the textbook Ventris & Chadwick result.

    print()
    print("cite it:", pylos.cite()[:90], "…")

# Also there when you want them: aegean.load("sigla") (the SigLA Linear A dataset),
# analysis.dispersions() (Gries' DP), and `aegean plot keyness damos --site Pylos -o out.png`.

### Interchange · EpiDoc in and out (offline)

The corpus model speaks **EpiDoc TEI**, the standard for digital epigraphy, in *both* directions with no extra dependency: `io.write_epidoc` exports, and `io.from_epidoc` reads any EpiDoc edition (a file or a folder of `.xml`) back into a corpus. So your own inscriptions, or any of the 400+ published EpiDoc corpora, round-trip through pyaegean entirely offline.

In [ ]:
import tempfile, os
from aegean import io as aio

# write the bundled Linear A 'Zakros' tablets to EpiDoc, then read them straight back;
# only the stdlib XML parser is used, no [epidoc] extra:
epi = os.path.join(tempfile.mkdtemp(), "epidoc")
aio.write_epidoc(aegean.load("lineara").filter(site="Zakros"), epi)
back = aio.from_epidoc(epi, script_id="lineara")

doc = back.documents[0]
print(f"read {len(back)} documents back from EpiDoc")
print(f"first: {doc.id} · {doc.meta.site} · {len(doc.tokens)} tokens")
print("tokens:", " ".join(t.text for t in doc.tokens[:6]))
# read 53 documents back from EpiDoc
# first: ZA10a · Zakros · 16 tokens
# tokens: TA-NA-TE 2 PA 1 A-KU-MI-NA 1
# From the shell: aegean import <dir> --epidoc --script lineara -o corpus.json

## Coda · Grounded, EXPLORATORY translation

pyaegean's AI layer (`aegean.ai` / `aegean.translate`) is **multi-provider** (Anthropic / OpenAI / Grok / Gemini / OpenRouter) and only generates after a **local, deterministic grounding** step builds evidence from the very tools above. The grounding needs no key or network, so we can inspect it here; the generative call needs a provider extra (e.g. `pyaegean[anthropic]`) and an API key, and is wrapped so a missing SDK or key is handled gracefully rather than crashing.

When the LSJ backend is switched on, that grounding also carries **gated dictionary glosses** (`translate(..., glosses=True)`, on by default): a polysemy gate leaves highly ambiguous words to the model, where forcing a single dictionary sense would mislead, and feeds a gloss only where a dominant sense genuinely helps. Pass `glosses=False` for lemma-only grounding.

> Any AI reading is a **hypothesis carrying its provenance**, explicitly labelled EXPLORATORY — never asserted truth. For undeciphered Linear A especially, it is a lead for a human expert.

In [ ]:
from aegean import ai, translate

print("providers:", ai.list_providers())   # ['anthropic', 'gemini', 'grok', 'openai', 'openrouter']

# The deterministic, LOCAL grounding evidence — derived from the pipeline above,
# no key or network required:
print(translate.grounding_for("μῆνιν ἄειδε θεά", "greek"))
# ['μῆνιν → lemma μῆνις', 'ἄειδε → lemma ἀείδω', 'θεά → lemma θεά']
print(translate.grounding_for("KU-RO KI-RO", "lineara"))
# ['KU-RO → /kuro/', 'KI-RO → /kiro/']

# The generative step is optional — it needs a provider SDK + an API key. Handled gracefully:
try:
    result = translate.translate("μῆνιν ἄειδε θεά", script="greek")  # default provider; reads its key from the env
    print(result.labeled())          # carries the [EXPLORATORY · …] tag + provenance
except (ai.ProviderNotInstalled, ai.MissingAPIKey) as e:
    print(f"Generative step skipped ({type(e).__name__}). The grounding above is fully local.")

## What you can do now

In one sitting you took **one line of Homer** down the whole ladder — syllables, accent, IPA, metre, POS, morphology — then, with a few opt-in calls, lifted POS/lemma to 100% on attested forms, *generated* lemmas for unseen ones with the neural model, glossed from the LSJ, and read a dependency tree. Then you turned the same toolkit on an **undeciphered** script to check a Bronze-Age ledger, quantify a minimal pair, mine a word-family, and measure a collocation — generating *leads*, not answers.

Every stage is an independent function, so you can lift just the rung you need onto **your own** text — or skip the ladder entirely: `greek.pipeline(text)` runs the whole stack in one call, `greek.use_neural_pipeline()` swaps in the most accurate model, `greek.load_work("tlg0012.tlg001")` fetches the complete Iliad as a corpus, `aegean.load("damos")` the full Linear B corpus, and `pip install "pyaegean[cli]"` gives you all of it from the command line (`aegean --help`).

**Where next**
- [Recipes](https://github.com/ryanpavlicek/pyaegean/wiki/Recipes) — end-to-end scholarly workflows: reconcile a whole corpus's accounting, map a word, lemmatize-and-cite a chapter, keyness, cross-script triage.
- [Greek NLP](https://github.com/ryanpavlicek/pyaegean/wiki/Greek-NLP) — the full reference (prosody, the benchmark harness, held-out and out-of-AGDT evaluation, and the opt-in backends in detail).
- [Analysis](https://github.com/ryanpavlicek/pyaegean/wiki/Analysis) — phonetic distance/alignment, morphological clustering, collocation statistics, the query engine.
- [Data & Provenance](https://github.com/ryanpavlicek/pyaegean/wiki/Data-and-Provenance) — what's bundled vs fetched, and the CC BY-SA attribution for the treebank and LSJ.

> The throughline: the toolkit tells you where it's confident and where it's guessing. For deciphered Greek that's an honest baseline you can upgrade to attested scholarship; for the undeciphered Linear A material it's always **EXPLORATORY** — leads for a human expert, never ground truth.